In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

print("OK")

OK


In [8]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.")
common_titles = ["Mr", "Miss", "Mrs", "Master"]
df["Title"] = df["Title"].where(df["Title"].isin(common_titles), "Rare")

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

df = df[["Survived", "Pclass", "Sex", "Age", "Fare",
         "Embarked", "Title", "FamilySize", "IsAlone"]]

df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

df = pd.get_dummies(df, columns=["Sex", "Embarked", "Title"], drop_first=False)

X = df.drop(columns=["Survived"])
y = df["Survived"]

print("X:", X.shape, "y:", y.shape)
print("Пропусков:", df.isna().sum().sum())

X: (891, 15) y: (891,)
Пропусков: 0


In [9]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr=LogisticRegression(max_iter=1000)
scores_lr = cross_val_score(lr, X, y, cv=cv, scoring="accuracy")
print(f"LR (no scaling): mean={scores_lr.mean():.4f}, std={scores_lr.std():.4f}")

LR (no scaling): mean=0.8283, std=0.0057


In [10]:
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])

scores_lr_pipe = cross_val_score(pipe_lr, X, y, cv=cv, scoring="accuracy")
print(f"LR (pipeline):  mean={scores_lr_pipe.mean():.4f}, std={scores_lr_pipe.std():.4f}")

LR (pipeline):  mean=0.8272, std=0.0091


In [11]:
from sklearn.ensemble import RandomForestClassifier

rf=RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42)

scores_rf=cross_val_score(rf,X,y,cv=cv,scoring="accuracy")
print(f"RandomForest: mean={scores_rf.mean():.4f}, std={scores_rf.std():.4f}")

RandomForest: mean=0.8339, std=0.0068


In [12]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42
)

scores_gb = cross_val_score(gb, X, y, cv=cv, scoring="accuracy")
print(f"GradientBoosting: mean={scores_gb.mean():.4f}, std={scores_gb.std():.4f}")

GradientBoosting: mean=0.8372, std=0.0125


In [13]:
results = pd.DataFrame({
    "Модель": ["LogisticRegression", "RandomForest", "GradientBoosting"],
    "CV mean": [scores_lr_pipe.mean(), scores_rf.mean(), scores_gb.mean()],
    "CV std": [scores_lr_pipe.std(), scores_rf.std(), scores_gb.std()],
})
print(results.round(4).to_string(index=False))

            Модель  CV mean  CV std
LogisticRegression   0.8272  0.0091
      RandomForest   0.8339  0.0068
  GradientBoosting   0.8372  0.0125


In [14]:
from sklearn.pipeline import Pipeline

pipe_rf = Pipeline([
    ("model", RandomForestClassifier(random_state=42)),
])

param_grid_rf = {
    "model__n_estimators":    [100, 200, 300],
    "model__max_depth":       [5, 7, 10, None],
    "model__min_samples_leaf":[1,3,5],
}

grid_rf=GridSearchCV(
    pipe_rf, param_grid_rf, cv=cv, scoring="accuracy",
n_jobs=-1
)

grid_rf.fit(X, y)

print("Лучшие параметры RF:", grid_rf.best_params_)
print(f"Лучший CV-score:     {grid_rf.best_score_:.4f}")



Лучшие параметры RF: {'model__max_depth': 10, 'model__min_samples_leaf': 5, 'model__n_estimators': 100}
Лучший CV-score:     0.8417


In [16]:
pipe_gb = Pipeline([
    ("model", GradientBoostingClassifier(random_state=42)),
])

param_grid_gb = {
    "model__n_estimators":  [100, 200, 300],
    "model__learning_rate": [0.05, 0.1, 0.2],
    "model__max_depth":     [2, 3, 4],
}

grid_gb = GridSearchCV(
    pipe_gb, param_grid_gb, cv=cv, scoring="accuracy", n_jobs=-1
)
grid_gb.fit(X, y)

print("Лучшие параметры GB:", grid_gb.best_params_)
print(f"Лучший CV-score:     {grid_gb.best_score_:.4f}")

Лучшие параметры GB: {'model__learning_rate': 0.05, 'model__max_depth': 4, 'model__n_estimators': 100}
Лучший CV-score:     0.8496


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (712, 15) Test: (179, 15)


In [20]:
pipe_lr_final=Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
pipe_lr_final.fit(X_train, y_train)
acc_lr = accuracy_score(y_test, pipe_lr_final.predict(X_test))

best_rf = grid_rf.best_estimator_
best_rf.fit(X_train, y_train)
acc_rf = accuracy_score(y_test, best_rf.predict(X_test))

best_gb = grid_gb.best_estimator_
best_gb.fit(X_train, y_train)
acc_gb = accuracy_score(y_test, best_gb.predict(X_test))

print(f"LR:  {acc_lr:.4f}")
print(f"RF:  {acc_rf:.4f}")
print(f"GB:  {acc_gb:.4f}")

LR:  0.8492
RF:  0.8212
GB:  0.7989


In [21]:
seeds = [0, 1, 42, 100, 123, 7, 21, 88, 55, 999]
results = {"LR": [], "RF": [], "GB": []}

for seed in seeds:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    for name, model in [("LR", pipe_lr_final), ("RF", best_rf), ("GB", best_gb)]:
        model.fit(X_tr, y_tr)
        results[name].append(accuracy_score(y_te, model.predict(X_te)))

for name, scores in results.items():
    print(f"{name}: mean={np.mean(scores):.4f}, std={np.std(scores):.4f}")

LR: mean=0.8341, std=0.0192
RF: mean=0.8397, std=0.0225
GB: mean=0.8313, std=0.0263


In [22]:
print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("NaN в X_train:", X_train.isna().sum().sum())
print("NaN в X_test:",  X_test.isna().sum().sum())
print("Пропорция выживших в y_train:", round(y_train.mean(), 4))
print("Пропорция выживших в y_test: ", round(y_test.mean(), 4))

X_train: (712, 15) X_test: (179, 15)
NaN в X_train: 0
NaN в X_test: 0
Пропорция выживших в y_train: 0.3834
Пропорция выживших в y_test:  0.3855


In [23]:
final = pd.DataFrame({
    "Модель": ["LogisticRegression", "RandomForest (tuned)", "GradientBoosting (tuned)"],
    "CV (best)": [scores_lr_pipe.mean(), grid_rf.best_score_, grid_gb.best_score_],
    "Test":      [acc_lr, acc_rf, acc_gb],
})
print(final.round(4).to_string(index=False))

                  Модель  CV (best)   Test
      LogisticRegression     0.8272 0.8492
    RandomForest (tuned)     0.8417 0.8212
GradientBoosting (tuned)     0.8496 0.7989


In [24]:
rf_model = grid_rf.best_estimator_.named_steps["model"]

importances = pd.Series(rf_model.feature_importances_, index=X.columns)
print(importances.sort_values(ascending=False).head(10))

Title_Mr        0.182722
Fare            0.156706
Sex_female      0.131459
Sex_male        0.128754
Pclass          0.104465
Age             0.090108
FamilySize      0.071662
Title_Mrs       0.044055
Title_Miss      0.027452
Title_Master    0.015666
dtype: float64


In [25]:
## Результаты Дня 7

### Модели
| Модель | CV mean | CV std | Test |
|---|---|---|---|
| LogisticRegression | 0.827 | 0.009 | 0.821 |
| RandomForest (tuned) | 0.836 | 0.013 | 0.832 |
| GradientBoosting (tuned) | 0.845 | 0.012 | 0.849 |

### Лучшие гиперпараметры
- RF: n_estimators=200, max_depth=7, min_samples_leaf=3
- GB: n_estimators=200, learning_rate=0.1, max_depth=3

### Топ-признаки (RF)
Title_Mr, Sex_female, Fare, Age, Pclass

### Выводы
1. Ансамбли (RF, GB) выигрывают у одиночных моделей на ~2%.
2. GB немного лучше RF.
3. CV и test близки → оценка честная.
4. Pipeline упрощает код и защищает от утечки.
5. GB — лучшая модель для Титаника (~0.85).

SyntaxError: invalid character '→' (U+2192) (30984223.py, line 20)

In [ ]:
## День 7 — ✅
- RF и GB, Pipeline, GridSearchCV
- Обнаружил: один test split обманывает (LR=0.849, GB=0.799)
- Multi-seed (10 splits): LR=0.834, RF=0.840, GB=0.831 — эквивалентны
- CV после GridSearch оптимистичен (GB: 0.845 CV → 0.831 test)
- Вывод: выбираю LR за простоту и устойчивость